# F1 Silver — Lap Times — SCD TYPE 2

**Existing SCD Type 1 logic is not changed.**

This notebook implements the same working SCD2 pattern used for Results.

**Business key:** `race_id, driver_id, lap`  
**Watermark:** `ingestion_date`  
**Target:** `formula1_<env>.silver.lap_times_scd2_demo`

```text
New       → INSERT
Unchanged → NO ACTION
Changed   → EXPIRE old + INSERT new
```


In [0]:
dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env")


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

CATALOG = f"formula1_{env}"
BRONZE = "bronze"
SILVER = "silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER}")

def bronze_table(name):
    return f"{CATALOG}.{BRONZE}.{name}"

def silver_table(name):
    return f"{CATALOG}.{SILVER}.{name}"


In [0]:
lap_times=spark.table(bronze_table("lap_times"))
lap_times.printSchema()
print("Bronze rows:",lap_times.count())


## 1. Incremental Batch

In [0]:
scd2_target_name = silver_table("lap_times_scd2_demo")

if not spark.catalog.tableExists(scd2_target_name):
    last_ts = None
else:
    last_ts = (
        spark.table(scd2_target_name)
        .agg(F.max("ingestion_date").alias("max_ts"))
        .first()["max_ts"]
    )

print("Last SCD2 ingestion_date:", last_ts)

lap_times_batch = (
    lap_times
    if last_ts is None
    else lap_times.filter(F.col("ingestion_date") > F.lit(last_ts))
)

print("Rows selected:", lap_times_batch.count())


## 2. Clean Selected Batch

In [0]:
lap_times_clean=(
    lap_times_batch
    .filter(F.col("race_id").isNotNull())
    .filter(F.col("driver_id").isNotNull())
    .filter(F.col("lap").isNotNull())
    .dropDuplicates(["race_id","driver_id","lap"])
    .withColumn("time_clean",F.trim("time"))
    .withColumn("milliseconds_clean",F.col("milliseconds").cast("long"))
    .withColumn("lap_position_category",
        F.when(F.col("position")==1,"Fastest")
         .when(F.col("position").between(2,10),"Top 10")
         .otherwise("Other"))
    .withColumn("silver_processed_timestamp",F.current_timestamp())
)
display(lap_times_clean.limit(20))

## 3. Create Record Hash

Technical ingestion/processing columns are excluded from the hash.

Therefore a new processing timestamp does not falsely create an SCD2 change.


In [0]:
business_columns = [
    c for c in lap_times_clean.columns
    if c not in [
        "ingestion_date",
        "silver_processed_timestamp",
        "effective_start_date",
        "effective_end_date",
        "is_current",
        "record_hash"
    ]
]

scd2_source = (
    lap_times_clean
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(F.col(c).cast("string"), F.lit("<NULL>"))
                    for c in business_columns
                ]
            ),
            256
        )
    )
)

print("Hash columns:", business_columns)


## 4. Initial SCD2 Load

In [0]:
if not spark.catalog.tableExists(scd2_target_name):

    batch_timestamp = spark.sql("SELECT current_timestamp()").first()[0]

    scd2_initial = (
        scd2_source
        .withColumn("effective_start_date", F.lit(batch_timestamp).cast("timestamp"))
        .withColumn("effective_end_date", F.lit(None).cast("timestamp"))
        .withColumn("is_current", F.lit(True))
    )

    (
        scd2_initial.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(scd2_target_name)
    )

    print("Initial SCD2 load completed.")
else:
    print("SCD2 target already exists.")


## 5. Detect New / Changed / Unchanged

In [0]:
current_target = (
    spark.table(scd2_target_name)
    .filter(F.col("is_current") == True)
    .select('race_id', 'driver_id', 'lap', "record_hash")
)

comparison = (
    scd2_source.alias("s").join(
        current_target.alias("t"),
        (F.col("s.race_id") == F.col("t.race_id"))
        & (F.col("s.driver_id") == F.col("t.driver_id"))
        & (F.col("s.lap") == F.col("t.lap")),
        "left"
    )
    .select(F.col("s.*"), F.col("t.record_hash").alias("target_hash"))
)

new_records = comparison.filter(F.col("target_hash").isNull())

changed_records = comparison.filter(
    F.col("target_hash").isNotNull()
    & (F.col("record_hash") != F.col("target_hash"))
)

unchanged_records = comparison.filter(
    F.col("target_hash").isNotNull()
    & (F.col("record_hash") == F.col("target_hash"))
)

print("New records      :", new_records.count())
print("Changed records  :", changed_records.count())
print("Unchanged records:", unchanged_records.count())


## 6. Build SCD2 Staging

For a changed row:

```text
EXPIRE row → matches current target
INSERT row → NULL merge key → does not match → inserts new version
```

For composite keys, separate `merge_key_0`, `merge_key_1`, etc. are used.


In [0]:
batch_timestamp = spark.sql("SELECT current_timestamp()").first()[0]
target_schema = spark.table(scd2_target_name).schema
target_columns = [field.name for field in target_schema.fields]

# EXPIRE rows
expire_stage = changed_records.select(
    F.col("race_id"), F.col("driver_id"), F.col("lap")
).withColumn("action", F.lit("EXPIRE"))

expire_stage = (
    expire_stage
    .withColumn("merge_key_0", F.col("race_id").cast("string"))
    .withColumn("merge_key_1", F.col("driver_id").cast("string"))
    .withColumn("merge_key_2", F.col("lap").cast("string"))
)


for field in target_schema.fields:
    if field.name not in expire_stage.columns:
        expire_stage = expire_stage.withColumn(field.name, F.lit(None).cast(field.dataType))

expire_stage = (
    expire_stage
    .withColumn("effective_end_date", F.lit(batch_timestamp).cast("timestamp"))
    .withColumn("is_current", F.lit(False))
)

# INSERT rows for changed records
insert_changed_stage = (
    changed_records
    .withColumn("effective_start_date", F.lit(batch_timestamp).cast("timestamp"))
    .withColumn("effective_end_date", F.lit(None).cast("timestamp"))
    .withColumn("is_current", F.lit(True))
    .withColumn("action", F.lit("INSERT"))
)

insert_changed_stage = (
    insert_changed_stage
    .withColumn("merge_key_0", F.lit(None).cast("string"))
    .withColumn("merge_key_1", F.lit(None).cast("string"))
    .withColumn("merge_key_2", F.lit(None).cast("string"))
)


# INSERT rows for new records
insert_new_stage = (
    new_records
    .withColumn("effective_start_date", F.lit(batch_timestamp).cast("timestamp"))
    .withColumn("effective_end_date", F.lit(None).cast("timestamp"))
    .withColumn("is_current", F.lit(True))
    .withColumn("action", F.lit("INSERT"))
)

insert_new_stage = (
    insert_new_stage
    .withColumn("merge_key_0", F.lit(None).cast("string"))
    .withColumn("merge_key_1", F.lit(None).cast("string"))
    .withColumn("merge_key_2", F.lit(None).cast("string"))
)

stage_columns = target_columns + [f"merge_key_{i}" for i in range(3)] + ["action"]

def align_stage(df):
    for field in target_schema.fields:
        if field.name not in df.columns:
            df = df.withColumn(field.name, F.lit(None).cast(field.dataType))
    return df.select(*stage_columns)

scd2_stage = (
    align_stage(expire_stage)
    .unionByName(align_stage(insert_changed_stage))
    .unionByName(align_stage(insert_new_stage))
)

print("SCD2 staging rows:", scd2_stage.count())


## 7. Single Delta MERGE — SCD Type 2

In [0]:
target = DeltaTable.forName(spark, scd2_target_name)

insert_values = {
    field.name: f"s.{field.name}"
    for field in spark.table(scd2_target_name).schema.fields
}

(
    target.alias("t")
    .merge(
        scd2_stage.alias("s"),
        "t.race_id = s.merge_key_0 AND t.driver_id = s.merge_key_1 AND t.lap = s.merge_key_2 AND t.is_current = true"
    )
    .whenMatchedUpdate(
        condition="s.action = 'EXPIRE'",
        set={
            "effective_end_date": "s.effective_end_date",
            "is_current": "s.is_current"
        }
    )
    .whenNotMatchedInsert(
        condition="s.action = 'INSERT'",
        values=insert_values
    )
    .execute()
)

print("LAP_TIMES SCD2 MERGE completed.")


## 8. Validation

In [0]:
scd2_result = spark.table(scd2_target_name)

print("Total rows   :", scd2_result.count())
print("Current rows :", scd2_result.filter(F.col("is_current") == True).count())
print("History rows :", scd2_result.filter(F.col("is_current") == False).count())

display(
    scd2_result.filter(F.col('milliseconds') == 118245).orderBy(
        'race_id', 'driver_id', 'lap',
        "effective_start_date"
    )
)


In [0]:
duplicate_current = (
    scd2_result
    .filter(F.col("is_current") == True)
    .groupBy('race_id', 'driver_id', 'lap')
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate current records:")
display(duplicate_current)
